In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
from model.autoencoder import Model
from train import Trainer
from data import get_cross_data, load_data


# Seed
torch.manual_seed(3)

In [2]:
cache_samples=False
save_samples=False
batch_size = 8
T = 64          # Number of frames (64)
M = 1           # Number of persons
V = 25          # Number of joints
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120' # 'ntu' or 'ntu120'
lr = 1e-4
train_samples = 64
test_samples = 32

In [3]:
if cache_samples:
    # Try to load paired data from pickle file
    if os.path.exists(f'data/{dataset}_{setting}_paired.pkl'):
        with open(f'data/{dataset}_{setting}_paired.pkl', 'rb') as f:
            paired_data = pickle.load(f)
            paired_train = paired_data['train']
            paired_test = paired_data['test']
            print('Paired data loaded from pickle file')
    else:
        # Load data
        X = load_data(dataset, T)
        # Generate paired data and save to pickle file
        paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, return_loader=True, train_samples=train_samples, test_samples=test_samples)
        if save_samples:
            with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
                pickle.dump({'train': paired_train, 'test': paired_test}, f)
                print('Paired data saved to pickle file')

else:
    # Load data
    X = load_data(dataset, T)
    paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, return_loader=True, train_samples=train_samples, test_samples=test_samples)

    if save_samples:
        with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
            pickle.dump({'train': paired_train, 'test': paired_test}, f)
            print('Paired data saved to pickle file')

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31


In [4]:
# Initialize the model
model = Model(num_class=120, num_point=V, num_person=M, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False, dataset=dataset)
model = model.cuda()

# Define optimizer
# Only optimize parameters that require gradients (unfrozen parameters)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

# Number of epochs
num_epochs = 10

# Create Trainer instance
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_paired_loader=paired_train,
    val_paired_loader=paired_test,
    num_epochs=num_epochs,
    wandb_project='Motion Retargeting',
    device='cuda', 
    dataset=dataset,
)

# Train
trainer.train()

# Save model
torch.save(model.state_dict(), 'model.pth')

e:\LocalCode\Transformer Retargeting\model\encoder.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_state_dict = torch.load(f'eval/mixformer/pretrained/{self.

Starting Training (Autoregressive Motion Retargeting)...
Epoch [1/10], Loss: 1.6290
{'mse': 0.23711350746452808, 'ee': 0.23861573869362473, 'smoothing': 0.96208905428648, 'inception': 0.1911412961781025}


KeyboardInterrupt: 